In [1]:
import sys 

sys.path.append("..")

In [15]:
from dehamer import Dehamer
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(1, 3, 224, 224).to(device)

model = Dehamer().to(device)
output = model(x)

print(f"Output shape is: {output.shape}")

/workspace/dehazing/.venv/lib/python3.11/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Output shape is: torch.Size([1, 3, 224, 224])


### Frequency and Spatial Dual-Guidance Network

In [4]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True, 
                 activation='prelu', norm=None):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, bias=bias)]
        
        if norm == 'batch':
            layers.append(nn.BatchNorm2d(out_ch))
        elif norm == 'instance':
            layers.append(nn.InstanceNorm2d(out_ch))

        acts = {
            'relu': nn.ReLU(True),
            'prelu': nn.PReLU(),
            'lrelu': nn.LeakyReLU(0.2, True),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid()
        }
        if activation in acts:
            layers.append(acts[activation])
        
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

Deconvolution (Convolution2d Transposed)

In [25]:
class DeconvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=True, 
                 activation='prelu', norm=None):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_ch, out_ch, kernel_size, stride, padding, bias=bias)]
        
        if norm == 'batch':
            layers.append(nn.BatchNorm2d(out_ch))
        elif norm == 'instance':
            layers.append(nn.InstanceNorm2d(out_ch))

        acts = {
            'relu': nn.ReLU(True),
            'prelu': nn.PReLU(),
            'lrelu': nn.LeakyReLU(0.2, True),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid()
        }
        if activation in acts:
            layers.append(acts[activation])
            
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

#### Unet Block

In [6]:
class UNetConvBlock(nn.Module):
    def __init__(self, in_chans, out_chans):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_chans, out_chans, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

In [7]:
class UNetUpBlock(nn.Module):
    def __init__(self, in_chans, out_chans, up_mode='upconv'):
        super().__init__()
        if up_mode == 'upconv':
            self.up = nn.ConvTranspose2d(in_chans, out_chans, kernel_size=2, stride=2)
        else:
            self.up = nn.Sequential(
                nn.Upsample(mode='bilinear', scale_factor=2, align_corners=False),
                nn.Conv2d(in_chans, out_chans, kernel_size=1)
            )
        self.conv_block = nn.Sequential(
            nn.Conv2d(out_chans * 2, out_chans, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, bridge):
        up = self.up(x)
        out = torch.cat([up, bridge], 1)
        return self.conv_block(out)

### Advanced Feature Block

In [35]:
class SAM(nn.Module):
    """Supervised Attention Module"""
    def __init__(self, n_feat, kernel_size = 1, bias = False):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv2d(n_feat, n_feat, kernel_size, padding=pad, bias=bias)
        self.conv2 = nn.Conv2d(n_feat, 3, kernel_size, padding=pad, bias=bias)
        self.conv3 = nn.Conv2d(3, n_feat, kernel_size, padding=pad, bias=bias)

    def forward(self, x, x_img):
        x1 = self.conv1(x)
        img = self.conv2(x) + x_img 

        x2 = torch.sigmoid(self.conv3(img))
        return (x1 * x2) + x, img
        

In [36]:
class ResBlock(nn.Module):
    def __init__(self, channel):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(channel, channel, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channel, channel, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv_1x1 = nn.Conv2d(channel, channel, kernel_size=1)

    def forward(self, x):
        return self.layers(x) + self.conv_1x1(x)

In [37]:
class ResBlock_fft_bench(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.main = nn.Conv2d(n_feat, n_feat, kernel_size=3, padding=1) 
        self.mag  = nn.Conv2d(n_feat, n_feat, kernel_size = 1)
        self.pha = nn.Sequential(
            nn.Conv2d(n_feat, n_feat, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(n_feat, n_feat, kernel_size=1),
        )

    def forward(self, x):
        _, _, H, W = x.shape
        fre = torch.fft.rfft2(x, norm = "backward")
        mag, phase = torch.abs(fre), torch.angle(fre)

        mag_out = self.mag(mag)

        # Phase modulation via softmax-weighted attention
        # Apply the attention on the haze correction in the Amplitude sections
        mag_res = mag_out - mag
        weight = F.softmax(F.adaptive_avg_pool2d(mag_res, (1, 1)), dim=1)
        # I know which frequency channels needed the most amplitude correction (from weight). 
        # Phase errors are probably concentrated in those same channels. 
        # So scale the phase by that attention before trying to correct it 
        # — this tells the network where to focus.
        phase_out = self.pha(phase * weight) + phase

        # After finding the correct phase and amplitude, we can recover back via 
        # inverse fourier transform (IVF)
        real = mag_out * torch.cos(phase_out)
        imag = mag_out * torch.sin(phase_out)
        fre_out = torch.complex(real, imag)
        y = torch.fft.irfft2(fre_out, s=(H, W), norm='backward')

        return self.main(x) + y

### MDC Blocks

In [38]:
class Decoder_MDCBlock1(nn.Module):
    def __init__(self, num_filter, num_ft, num, kernel_size=4, stride=2, padding=1):
        super().__init__()
        self.num_ft = num_ft - 1
        self.down_convs = nn.ModuleList()
        self.up_convs = nn.ModuleList()
        curr_ch = num_filter
        for i in range(self.num_ft):
            next_ch = curr_ch + 2 ** (num + i)
            self.down_convs.append(
                ConvBlock(curr_ch, next_ch, kernel_size,stride, padding)
            )
            self.up_convs.append(
                DeconvBlock(next_ch, curr_ch, kernel_size, stride, padding)
            )
            curr_ch = next_ch

    def forward(self, ft_h, ft_l_list):
        ft_fusion = ft_h
        for i, skip_ft in enumerate(ft_l_list):
            ft = ft_fusion
            depth = self.num_ft - i
            for j in range(depth):
                ft = self.down_convs[j](ft)
            
            ft = F.interpolate(ft, size=skip_ft.shape[-2:], mode='bilinear', align_corners=False)
            ft = ft - skip_ft
            
            for j in range(depth):
                ft = self.up_convs[depth - j - 1](ft)
                
            ft_fusion = F.interpolate(ft_fusion, size=ft.shape[-2:], mode='bilinear', align_corners=False)
            ft_fusion = ft_fusion + ft
        return ft_fusion

In [39]:
class Encoder_MDCBlock1(nn.Module):
    def __init__(self, num_filter, num_ft, kernel_size = 4, stride = 2, padding = 1):
        super().__init__()
        self.num_ft = num_ft - 1
        self.up_convs = nn.ModuleList()
        self.down_convs = nn.ModuleList()
        curr_ch = num_filter
        for i in range(self.num_ft):
            next_ch = curr_ch - 2 ** (num_ft - i)
            self.up_convs.append(
                DeconvBlock(curr_ch, next_ch, kernel_size, stride, padding)
            )
            self.down_convs.append(
                ConvBlock(next_ch, curr_ch, kernel_size, stride, padding)
            )
            curr_ch = next_ch

    def forward(self, ft_l, ft_h_list):
        ft_fusion = ft_l
        for i, skip_ft in enumerate(ft_h_list):
            ft = ft_fusion
            depth = self.num_ft - i
            for j in range(depth):
                ft = self.up_convs[j](ft)
            
            ft = F.interpolate(ft, size=skip_ft.shape[-2:], mode='bilinear', align_corners=False)
            ft = ft - skip_ft
            
            for j in range(depth):
                ft = self.down_convs[depth - j - 1](ft)
            
            ft_fusion = F.interpolate(ft_fusion, size=ft.shape[-2:], mode='bilinear', align_corners=False)
            ft_fusion = ft_fusion + ft
        return ft_fusion

### FSDGN Model

In [40]:
class FSDGN(nn.Module):
    def __init__(self, num_in_ch=3, base_channel=16, up_mode='upconv', bias=False):
        super().__init__()
        
        # Channels: [16, 20, 28, 44, 76]
        chs = [base_channel, 20, 28, 44, 76]
        # Channels sequence for Encoder -> Bottleneck -> Decoder
        # E.g [16, 20, 28, 44, 76, 44, 28, 20, 16] 
        chs_enc_dec = chs + chs[-2::-1]

        # ------------------- STAGE 1 (Frequency / Global Branch) -------------------
        self.enc_convs = nn.ModuleList([nn.Conv2d(num_in_ch, chs[0], 3, 1, 1)])
        self.enc_convs.extend([UNetConvBlock(chs[i], chs[i + 1]) for i in range(4)])
        self.dec_convs = nn.ModuleList([UNetUpBlock(chs[i + 1], chs[i], up_mode) for i in range(4)])
        
        # Frequency and Residual Blocks
        self.res_blocks = nn.ModuleList([ResBlock(c) for c in chs_enc_dec])
        self.fft_blocks = nn.ModuleList([ResBlock_fft_bench(c) for c in chs_enc_dec])
        
        # Fusion Blocks (
        self.enc_fusions = nn.ModuleList([
            Encoder_MDCBlock1(chs[1], 2), Encoder_MDCBlock1(chs[2], 3),
            Encoder_MDCBlock1(chs[3], 4), Encoder_MDCBlock1(chs[4], 5)
        ])
        self.dec_fusions = nn.ModuleList([
            Decoder_MDCBlock1(chs[3], 2, 5), Decoder_MDCBlock1(chs[2], 3, 4),
            Decoder_MDCBlock1(chs[1], 4, 3), Decoder_MDCBlock1(chs[0], 5, 2)
        ])

        # ------------------- STAGE 2 (Spatial / Local Branch) -------------------
        self.enc_convs2 = nn.ModuleList([nn.Conv2d(num_in_ch, chs[0], 3, 1, 1)])
        self.enc_convs2.extend([UNetConvBlock(chs[i], chs[i+1]) for i in range(4)])
        self.dec_convs2 = nn.ModuleList([UNetUpBlock(chs[i+1], chs[i], up_mode) for i in range(4)])
        self.res_blocks2 = nn.ModuleList([ResBlock(c) for c in chs_enc_dec])
        
        self.enc_fusions2 = nn.ModuleList([
            Encoder_MDCBlock1(chs[1], 2), Encoder_MDCBlock1(chs[2], 3),
            Encoder_MDCBlock1(chs[3], 4), Encoder_MDCBlock1(chs[4], 5)
        ])
        self.dec_fusions2 = nn.ModuleList([
            Decoder_MDCBlock1(chs[3], 2, 5), Decoder_MDCBlock1(chs[2], 3, 4),
            Decoder_MDCBlock1(chs[1], 4, 3), Decoder_MDCBlock1(chs[0], 5, 2)
        ])

        # Cross-Stage Feature Fusion (CSFF)
        self.csff_enc = nn.ModuleList([nn.Conv2d(c, c, 1, bias=bias) for c in chs[:-1]])
        self.csff_dec = nn.ModuleList([nn.Conv2d(c, c, 1, bias=bias) for c in chs[:-1]])

        self.sam = SAM(chs[0], kernel_size=1)
        self.concat = nn.Conv2d(chs[0] * 2, chs[0], 3, padding=1)
        self.last = nn.Conv2d(chs[0], num_in_ch, kernel_size=1)

    def forward(self, x):
        identity = x
        
        # ================= STAGE 1: Frequency Guided Branch =================
        enc_feats = []
        out = self.enc_convs[0](x)
        out = self.fft_blocks[0](self.res_blocks[0](out))
        enc_feats.append(out)

        # Stage 1 Encoder 
        for i in range(4):
            out = self.enc_convs[i + 1](out)
            out = self.enc_fusions[i](out, enc_feats)
            out = self.fft_blocks[i + 1](self.res_blocks[i + 1](out))
            enc_feats.append(out)

        # Stage 1 Decoder
        dec_feats = [enc_feats[-1]]
        curr_out = enc_feats[-1]
        for i in range(4):
            idx = 3 - i # 3, 2, 1, 0 (Reversing back up the U-Net)
            # print("Curr_out: ", curr_out.shape)
            # print(f"Enc_feats at {idx}: {enc_feats[idx].shape}")
            
            curr_out = self.dec_convs[idx](curr_out, enc_feats[idx])
            
            # 5+i maps to [44, 28, 20, 16] in the chs_enc_dec list
            curr_out = self.res_blocks[5 + i](curr_out)
            curr_out = self.fft_blocks[5 + i](curr_out)
            curr_out = self.dec_fusions[i](curr_out, dec_feats)
            
            dec_feats.append(curr_out)

        sam_feats, stage1_img = self.sam(dec_feats[-1], identity)

        # ================= STAGE 2: Spatial Guided Branch =================
        enc_feats2 = []
        
        out_2 = self.enc_convs2[0](identity)
        y_concat = self.concat(torch.cat([out_2, sam_feats], dim = 1))
        
        # CSFF from Stage 1: dec_feats[-1] is the last output of Stage 1 decoder
        y = self.res_blocks2[0](y_concat)
        y = y + self.csff_enc[0](enc_feats[0]) + self.csff_dec[0](dec_feats[-1])
        enc_feats2.append(y)

        # --- Encoder Stage 2 ---
        for i in range(4):
            y = self.enc_convs2[i + 1](y)
            y = self.enc_fusions2[i](y, enc_feats2)
            y = self.res_blocks2[i + 1](y)
            if i < 3: # Apply CSFF skip connections
                y = y + self.csff_enc[i + 1](enc_feats[i + 1]) + \
                        self.csff_dec[i + 1](dec_feats[-(i + 2)])
            enc_feats2.append(y)

        # Decoder Stage 2
        dec_feats2 = [enc_feats2[ -1 ]]
        curr_y = enc_feats2[ -1 ]
        for i in range(4):
            idx = 3 - i
            curr_y = self.dec_convs2[idx](curr_y, enc_feats2[idx])
            curr_y = self.dec_fusions2[i](self.res_blocks2[5 + i](curr_y), dec_feats2)
            dec_feats2.append(curr_y)

        final_out = torch.clamp(self.last(dec_feats2[-1]), 0, 1)
        stage1_img = torch.clamp(stage1_img, 0, 1)

        return final_out, stage1_img

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(1, 3, 224, 224).to(device)

model = FSDGN().to(device)
output, _ = model(x)

print(f"Output shape is: {output.shape}")

Output shape is: torch.Size([1, 3, 224, 224])


## FDSGN testing

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from fsdgn import FSDGN

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(1, 3, 224, 224).to(device)

model = FSDGN().to(device)
output, _ = model(x)

print(f"Output shape is: {output.shape}")

Output shape is: torch.Size([1, 3, 224, 224])
